## Databricks Data Pipeline

This Databricks notebook provides a solution to the task detailed in the T-SQL_workflow code. This notebook aims to highlight some of the common changes required when adapting code used in SQL Server to run in Databricks. It may be helpful to look at this example alongside the T-SQL code to compare the differences in syntax.

Pipeline stages:
1. **Check data** - Look at data structure of Autumn 2025 pupils and schools data
2. **Data Manipulation** - Select columns and parse metadata, join pupils to schools
3. **Outputs** - Produce average pupil premium by Ofsted rating summary

In [0]:
-- Define catalog being used
USE CATALOG catalog_40_copper_analyst_training;

### Check data
Look at data structure. Note here that `SELECT TOP 1000 * FROM [table]` becomes `SELECT * FROM table LIMIT 1000` - this is the first of several syntax changes that are required when moving from working with SQL in SQL Server Management Studio to Databricks.

In [0]:
-- Check structure of pupils_autumn_2025 table
SELECT * FROM catalog_40_copper_analyst_training.bronze.pupils_autumn_2025 LIMIT 1000

In [0]:
-- Check structure of schools_autumn_2025 table
SELECT * FROM catalog_40_copper_analyst_training.bronze.schools_autumn_2025 LIMIT 1000

### Data Manipulation
Select required columns and parse JSON metadata, join pupils to schools.

Notice that, instead of creating new objects using `SELECT col_name INTO new_object FROM old_object` (often with the familiar `DROP TABLE new_object IF EXISTS`, or `IF OBJECT_ID('tempdb..#new_table') IS NOY NULL DROP TABLE #new_table`), we use the syntax `CREATE OR REPLACE <OBJECT TYPE> new_object AS`, before defining the new object with a SELECT statement. In the example below we demonstrate the use of temporary views, and temporary tables, though same process is used for creating permanent objects if the TEMP / TEMPORARY keyword is removed (as long as the user has write access to the catalog, which we don't here!). 

Note also the difference in syntax within the select statement and simplification of JSON processing compared to the T-SQL example.

In [0]:
-- Pupils table: select columns and parse JSON metadata
CREATE OR REPLACE TEMPORARY VIEW pupils_aut AS
SELECT 
  pupil_id,
  gender,
  school_urn,
  metadata_json:address:postcode AS postcode,
  LOWER(metadata_json:sen_status) AS sen_status,
  metadata_json:fsm_eligible AS fsm_status
FROM catalog_40_copper_analyst_training.bronze.pupils_autumn_2025;

-- Schools table: select columns, parse JSON, reformat data in city column, deduplicate
CREATE OR REPLACE TEMP VIEW schools_aut AS -- We use TEMP here instead of TEMPORARY in the above example - these can be used interchangeably
SELECT DISTINCT
  school_urn,
  INITCAP(city) AS city,
  school_type,
  REPLACE(metadata_json:ofsted_rating, '"', '') AS ofsted_rating,
  CAST(metadata_json:pupil_premium_pct AS DOUBLE) AS pupil_premium_pct
FROM catalog_40_copper_analyst_training.bronze.schools_autumn_2025;

-- Join pupils and schools tables ahead of performing calculations
CREATE OR REPLACE TEMP TABLE pupils_schools_aut AS -- and here we create a temporary table rather than a view
SELECT 
  p.*,
  s.city,
  s.school_type,
  s.ofsted_rating,
  s.pupil_premium_pct
FROM pupils_aut p
LEFT JOIN schools_aut s
  ON p.school_urn = s.school_urn

In [0]:
-- Investigate relationships in joined data
SELECT gender, ofsted_rating, COUNT(*) AS n
FROM pupils_schools_aut
GROUP BY gender, ofsted_rating
ORDER BY n DESC

### Outputs
Produce summary output: calculate average pupil premium percentage by Ofsted rating. 

Note that the query here does not require a common table expression, whereas the T-SQL equivalent did. This is an example of a function that is available in SparkSQL but not in T-SQL, which allows code to be simplified when transitioning between platforms (in this case the PERCENTILE aggregate function, which is not available in T-SQL). 

In [0]:
-- Create final output table (note that we are only creating a temporary table here due to lack of write access to catalog)
CREATE OR REPLACE TEMP TABLE avg_pupil_premium_by_ofsted AS
SELECT 
  ofsted_rating,
  COUNT(*) AS pupils,
  ROUND(AVG(pupil_premium_pct), 4) AS avg_pupil_premium_pct,
  ROUND(PERCENTILE(pupil_premium_pct, 0.5), 4) AS median_pupil_premium_pct
FROM pupils_schools_aut
GROUP BY ofsted_rating
ORDER BY pupils DESC;

-- View final table
SELECT *
FROM avg_pupil_premium_by_ofsted

Note also that a more direct translation of the T-SQL code can also be run in Databricks as long as the square brackets are removed and table references are updated (demonstrated below). The above example was chosen to show some of the more substantive differences between platforms, as there are sometimes not direct equivalents for every function, with more significant sometimes needed.

In [0]:
%sql
-- Common Table Expression to calculate median pupil premium 
CREATE OR REPLACE TEMP TABLE avg_pupil_premium_by_ofsted AS
WITH median_pp_prep AS (
    SELECT
        ofsted_rating,
        pupil_premium_pct,
        PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY pupil_premium_pct)
            OVER (PARTITION BY ofsted_rating) AS median_pupil_premium_pct
    FROM pupils_schools_aut
)
-- Create final output table
SELECT
    ofsted_rating,
    COUNT(*) AS pupils,
    ROUND(AVG(pupil_premium_pct), 4) AS avg_pupil_premium_pct,
    ROUND(MAX(median_pupil_premium_pct), 4) AS median_pupil_premium_pct
FROM median_pp_prep
GROUP BY ofsted_rating
ORDER BY pupils DESC;

-- View final table
SELECT *
FROM avg_pupil_premium_by_ofsted